# This notebook implements the first preprocessing component: general text cleaning.
## It works step by step and shows the output after each transformation.

# Step 1 — Imports

In [11]:
import json
import re
import sys
import unicodedata
from pathlib import Path

print("Imports loaded successfully.")

Imports loaded successfully.


# Step 2  Connect to MongoDB
## The notebook supports both common locations:
### - running from the repository root
### - running from app/preprocessing/

In [12]:
from pathlib import Path
import sys

# Find the project root
cwd = Path.cwd().resolve()

search_locations = [cwd, *cwd.parents]

PROJECT_ROOT = next(
    (
        path for path in search_locations
        if (path / "app" / "mongo" / "mongodb.py").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root. "
        "Expected app/mongo/mongodb.py to exist."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.mongo.mongodb import db

# Test MongoDB connection
db.command("ping")

print("MongoDB connection successful")
print("Database:", db.name)
print("Project root:", PROJECT_ROOT)

MongoDB connection successful
Database: medical_diagnosis_ai
Project root: C:\Users\po\Downloads\medical-diagnosis


# Step 3  Load 406 conditions from MongoDB

In [13]:
conditions_collection = db["conditions"]

conditions = list(
    conditions_collection.find(
        {},
        {"_id": 0}
    )
)

EXPECTED_FIELDS = {
    "condition",
    "symptoms",
    "causes",
    "warnings",
    "recommendations",
}

assert isinstance(conditions, list), "MongoDB result must be a list."

assert len(conditions) > 0, (
    "No conditions were found in MongoDB."
)

invalid_records = [
    record
    for record in conditions
    if set(record.keys()) != EXPECTED_FIELDS
]

assert not invalid_records, (
    f"{len(invalid_records)} record(s) do not match "
    "the expected 5-field schema."
)

print(f"Loaded records from MongoDB: {len(conditions)}")
print("Collection:", conditions_collection.name)
print("Schema validation: PASSED")
print("Fields:", sorted(EXPECTED_FIELDS))

Loaded records from MongoDB: 406
Collection: conditions
Schema validation: PASSED
Fields: ['causes', 'condition', 'recommendations', 'symptoms', 'warnings']


# Step 4 — Select a real record for before/after inspection
## We use Acute pancreatitis because it contains realistic multiline medical text.

In [14]:
sample_record = next(
    record for record in conditions
    if record["condition"] == "Acute pancreatitis"
)

print("Condition:", sample_record["condition"])
print("\nRAW SYMPTOMS (first 800 characters):\n")
print(sample_record["symptoms"][:800])

Condition: Acute pancreatitis

RAW SYMPTOMS (first 800 characters):

The main symptom of acute pancreatitis is a severe, dull pain around the top of your stomach that develops suddenly.
This aching pain often gets steadily worse and can travel along your back or below your left shoulder blade. Eating or drinking may also make you feel worse very quickly, especially fatty foods.
Leaning forward or curling into a ball may help to relieve the pain, but lying flat on your back often increases the pain.
Acute pancreatitis caused by gallstones usually develops after eating a large meal. If the condition is caused by alcohol, the pain often develops 6 to 12 hours after drinking a significant amount of alcohol.

Other symptoms of acute pancreatitis can include:
- nausea (feeling sick) or vomiting
- diarrhoea
- indigestion
- a high temperature (fever) of 38C (100.4F


# Step 5 — Normalize Unicode characters

In [15]:
def normalize_unicode(text: str) -> str:
    if not text:
        return ""

    text = unicodedata.normalize("NFC", text)    

    replacements = {
        "\u2018": "'",
        "\u2019": "'",
        "\u201c": '"',
        "\u201d": '"',
        "\u2013": "-",
        "\u2014": "-",
        "\u00a0": " ",
        "\u2026": "...",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    return text


unicode_demo = "Patient’s pain — severe…\u00a0and worsening."
print("Before:", unicode_demo)
print("After: ", normalize_unicode(unicode_demo))

Before: Patient’s pain — severe… and worsening.
After:  Patient's pain - severe... and worsening.


# Step 6 — Normalize line endings

In [16]:
def normalize_line_endings(text: str) -> str:
    if not text:
        return ""

    return text.replace("\r\n", "\n").replace("\r", "\n")


line_demo = "fever\r\nnausea\rvomiting"
print("Before repr:", repr(line_demo))
print("After repr: ", repr(normalize_line_endings(line_demo)))

Before repr: 'fever\r\nnausea\rvomiting'
After repr:  'fever\nnausea\nvomiting'


# Step 7 — Normalize bullet formatting
## Bullet symbols are removed, but the medical content stays on separate lines.

In [17]:
def normalize_bullets(text: str) -> str:
    if not text:
        return ""

    cleaned_lines = []

    for line in text.splitlines():
        line = line.strip()

        if not line:
            continue

        line = re.sub(
            r"^(?:[•●▪◦‣⁃*-]|\d+[.)])\s*",
            "",
            line,
        )

        cleaned_lines.append(line.strip())

    return "\n".join(cleaned_lines)


bullet_demo = '''Symptoms include:
- severe abdominal pain
• nausea
* vomiting
1. fever
'''

print("Before:\n")
print(bullet_demo)
print("After:\n")
print(normalize_bullets(bullet_demo))

Before:

Symptoms include:
- severe abdominal pain
• nausea
* vomiting
1. fever

After:

Symptoms include:
severe abdominal pain
nausea
vomiting
fever


# Step 8 — Remove known website/UI noise
## The removal rules are deliberately conservative. We only remove phrases that are  clearly non-medical website boilerplate.

In [18]:
NOISE_PATTERNS = [
    r"help us improve nhs inform[^.!?]*[.!?]?",
    r"thank you for your feedback[^.!?]*[.!?]?",
    r"this video requires marketing cookies[^.!?]*[.!?]?",
    r"update your consent[^.!?]*[.!?]?",
    r"opens in (?:a )?new browser window[^.!?]*[.!?]?",
    r"source:\s*[^\n]*",
    r"last updated:\s*[^\n]*",

    # External/reference text
    r"Macmillan Cancer Support have more information[^.!?]*[.!?]",
    r"The images on this page were sourced from[^\n]*"

    # NHS navigation / cross-page text
    r"There's a separate page if[^.!?]*[.!?]",
    r"You can also get advice about[^.!?]*[.!?]",
]


def remove_noise(text: str) -> str:
    if not text:
        return ""

    for pattern in NOISE_PATTERNS:
        text = re.sub(
            pattern,
            "",
            text,
            flags=re.IGNORECASE | re.MULTILINE,
        )

    # NHS cross-page navigation sentence
    text = re.sub(
        r"There's a separate page[^.]*\.",
        "",
        text,
        flags=re.IGNORECASE,
    )

    # Image/source attribution sentence
    text = re.sub(
        r"The images on this page were sourced from\s+\S+\s*\.",
        "",
        text,
        flags=re.IGNORECASE,
    )

    return text.strip()


noise_demo = '''Severe chest pain
Help us improve NHS inform
Shortness of breath
Last updated: 10 October 2026
'''

print("Before:\n")
print(noise_demo)
print("After:\n")
print(remove_noise(noise_demo))

Before:

Severe chest pain
Help us improve NHS inform
Shortness of breath
Last updated: 10 October 2026

After:

Severe chest pain


# Step 9 — Normalize whitespace

In [19]:
def normalize_whitespace(text: str) -> str:
    if not text:
        return ""

    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n[ \t]+", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


whitespace_demo = "  severe    abdominal pain\n\n\n   nausea   "
print("Before repr:", repr(whitespace_demo))
print("After repr: ", repr(normalize_whitespace(whitespace_demo)))

Before repr: '  severe    abdominal pain\n\n\n   nausea   '
After repr:  'severe abdominal pain\n\nnausea'


# Step 10 — Remove exact duplicate lines

In [20]:
def remove_duplicate_lines(text: str) -> str:
    if not text:
        return ""

    seen = set()
    result = []

    for line in text.splitlines():
        cleaned = line.strip()

        if not cleaned:
            continue

        key = cleaned.casefold()

        if key not in seen:
            seen.add(key)
            result.append(cleaned)

    return "\n".join(result)


duplicate_line_demo = '''fatigue
shortness of breath
Fatigue
headache
shortness of breath
'''

print("Before:\n")
print(duplicate_line_demo)
print("After:\n")
print(remove_duplicate_lines(duplicate_line_demo))

Before:

fatigue
shortness of breath
Fatigue
headache
shortness of breath

After:

fatigue
shortness of breath
headache


# Step 11 — Remove exact duplicate sentences
## Only exact case-insensitive duplicate sentences are removed. We intentionally avoid aggressive fuzzy deletion because similar medical sentences may contain different information.

In [21]:
def remove_duplicate_sentences(text: str) -> str:
    if not text:
        return ""

    parts = re.split(r"(?<=[.!?])\s+", text.strip())

    if len(parts) <= 1:
        return text.strip()

    seen = set()
    result = []

    for sentence in parts:
        sentence = sentence.strip()

        if not sentence:
            continue

        key = sentence.casefold()

        if key not in seen:
            seen.add(key)
            result.append(sentence)

    return " ".join(result)


duplicate_sentence_demo = (
    "Seek medical help if symptoms become severe. "
    "Drink enough fluids. "
    "Seek medical help if symptoms become severe."
)

print("Before:")
print(duplicate_sentence_demo)
print("\nAfter:")
print(remove_duplicate_sentences(duplicate_sentence_demo))

Before:
Seek medical help if symptoms become severe. Drink enough fluids. Seek medical help if symptoms become severe.

After:
Seek medical help if symptoms become severe. Drink enough fluids.


# Step 12 — Combine the operations into clean_text()
## This is the human-readable cleaning function used for:
### - symptoms
### - causes
### - warnings
### - recommendations
## It aims to preserve medical meaning.

In [22]:
def clean_text(text: str) -> str:
    if not text:
        return ""

    text = normalize_unicode(text)
    text = normalize_line_endings(text)
    text = remove_noise(text)
    text = normalize_bullets(text)
    text = normalize_whitespace(text)
    text = remove_duplicate_lines(text)
    text = remove_duplicate_sentences(text)
    text = normalize_whitespace(text)

    return text.strip()


pipeline_demo = '''Symptoms may include:

• Severe   abdominal pain
- Nausea
- Vomiting
- Vomiting

Help us improve NHS inform
'''

print("RAW:\n")
print(pipeline_demo)
print("\nCLEANED:\n")
print(clean_text(pipeline_demo))

RAW:

Symptoms may include:

• Severe   abdominal pain
- Nausea
- Vomiting
- Vomiting

Help us improve NHS inform


CLEANED:

Symptoms may include:
Severe abdominal pain
Nausea
Vomiting


# Step 13 — Create clean_for_model()
## The model version:
### - starts from the safe human-readable cleaner
### - converts text to lowercase
### - removes most punctuation
### - keeps apostrophes and hyphens
### - does not remove stop words
### - keeps negation words such as no, not, and without
### - does not stem or lemmatize yet

In [23]:
def clean_for_model(text: str, lowercase: bool = True) -> str:
    text = clean_text(text)

    # Remove trailing external-reference blocks from symptom model text
    text = re.sub(
        r"Macmillan Cancer Support have (?:more )?information[\s\S]*$",
        "",
        text,
        flags=re.IGNORECASE,
    )
    
    if lowercase:
        text = text.lower()

    text = re.sub(r"[^\w\s'-]", " ", text, flags=re.UNICODE)
    text = text.replace("_", " ")
    text = re.sub(r"\s+", " ", text)

    return text.strip()


model_demo = "No severe chest pain; WITHOUT vomiting, but shortness-of-breath."
print("Readable:")
print(clean_text(model_demo))
print("\nModel version:")
print(clean_for_model(model_demo))

Readable:
No severe chest pain; WITHOUT vomiting, but shortness-of-breath.

Model version:
no severe chest pain without vomiting but shortness-of-breath


# Step 14 — Apply the complete cleaner to a real NHS record
## Notice that the original record is not changed. We create derived cleaned values.

In [24]:
real_symptoms_raw = sample_record["symptoms"]
real_symptoms_clean = clean_text(real_symptoms_raw)
real_symptoms_model = clean_for_model(real_symptoms_raw)

print("CONDITION:")
print(sample_record["condition"])

print("\n--- RAW (first 700 chars) ---")
print(real_symptoms_raw[:700])

print("\n--- CLEANED / READABLE (first 700 chars) ---")
print(real_symptoms_clean[:700])

print("\n--- MODEL VERSION (first 700 chars) ---")
print(real_symptoms_model[:700])

CONDITION:
Acute pancreatitis

--- RAW (first 700 chars) ---
The main symptom of acute pancreatitis is a severe, dull pain around the top of your stomach that develops suddenly.
This aching pain often gets steadily worse and can travel along your back or below your left shoulder blade. Eating or drinking may also make you feel worse very quickly, especially fatty foods.
Leaning forward or curling into a ball may help to relieve the pain, but lying flat on your back often increases the pain.
Acute pancreatitis caused by gallstones usually develops after eating a large meal. If the condition is caused by alcohol, the pain often develops 6 to 12 hours after drinking a significant amount of alcohol.

Other symptoms of acute pancreatitis can include:
- nau

--- CLEANED / READABLE (first 700 chars) ---
The main symptom of acute pancreatitis is a severe, dull pain around the top of your stomach that develops suddenly. This aching pain often gets steadily worse and can travel along your back o

# Step 15 — Clean all four text fields for all 406 conditions
## condition remains the class label.
## For each source record we produce:
### - cleaned symptoms
### - cleaned causes
### - cleaned warnings
### - cleaned recommendations
### - symptoms_model for later NLP stages
## The MongoDB/source dataset is not overwritten.

In [25]:
def clean_condition_record(record: dict) -> dict:
    return {
        "condition": record["condition"].strip(),
        "symptoms": clean_text(record.get("symptoms", "")),
        "causes": clean_text(record.get("causes", "")),
        "warnings": clean_text(record.get("warnings", "")),
        "recommendations": clean_text(record.get("recommendations", "")),
        "symptoms_model": clean_for_model(record.get("symptoms", "")),
    }


cleaned_conditions = [
    clean_condition_record(record)
    for record in conditions
]

print(f"Source records:  {len(conditions)}")
print(f"Cleaned records: {len(cleaned_conditions)}")
print("\nExample cleaned record fields:")
print(list(cleaned_conditions[0].keys()))

Source records:  406
Cleaned records: 406

Example cleaned record fields:
['condition', 'symptoms', 'causes', 'warnings', 'recommendations', 'symptoms_model']


# Step 16 — Validate the text-cleaning output

In [26]:
condition_names = [r["condition"] for r in cleaned_conditions]

validation = {
    "total_records": len(cleaned_conditions),
    "duplicate_condition_names": len(condition_names) - len(set(condition_names)),
    "empty_condition": sum(not r["condition"] for r in cleaned_conditions),
    "empty_symptoms": sum(not r["symptoms"] for r in cleaned_conditions),
    "empty_symptoms_model": sum(not r["symptoms_model"] for r in cleaned_conditions),
    "empty_causes": sum(not r["causes"] for r in cleaned_conditions),
    "empty_warnings": sum(not r["warnings"] for r in cleaned_conditions),
    "empty_recommendations": sum(not r["recommendations"] for r in cleaned_conditions),
}

for key, value in validation.items():
    print(f"{key}: {value}")

assert validation["total_records"] == len(conditions)
assert validation["duplicate_condition_names"] == 0
assert validation["empty_condition"] == 0
assert validation["empty_symptoms"] == 0
assert validation["empty_symptoms_model"] == 0

print("\nCore validation: PASSED")

total_records: 406
duplicate_condition_names: 0
empty_condition: 0
empty_symptoms: 0
empty_symptoms_model: 0
empty_causes: 83
empty_warnings: 131
empty_recommendations: 36

Core validation: PASSED


# Step 17 — Check that important negation words are preserved
## This matters because removing words such as no, not, or without can reverse medical meaning.

In [27]:
negation_tests = [
    "no fever",
    "not breathing normally",
    "without vomiting",
]

for text in negation_tests:
    cleaned = clean_for_model(text)
    print(f"{text!r} -> {cleaned!r}")

'no fever' -> 'no fever'
'not breathing normally' -> 'not breathing normally'
'without vomiting' -> 'without vomiting'


# Step 18 — Save the intermediate text-cleaned dataset
## When this notebook is run inside your project, the file will normally be saved to:
## app/dataset/processed/conditions_text_cleaned.json
## This is an intermediate preprocessing artifact. The original MongoDB conditions collection remains untouched

In [28]:
OUTPUT_DIR = PROJECT_ROOT / "app" / "dataset" / "output" / "preprocessing"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "conditions_text_cleaned.json"

with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(
        cleaned_conditions,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("Saved cleaned dataset:")
print(OUTPUT_PATH)
print(f"Records saved: {len(cleaned_conditions)}")

Saved cleaned dataset:
C:\Users\po\Downloads\medical-diagnosis\app\dataset\output\preprocessing\conditions_text_cleaned.json
Records saved: 406


# Step 19 — Final text-cleaning summary
## At this point:

### 406 source conditions
   ###    ↓
### Unicode normalization
 ###    ↓
### Line-ending normalization
 ###       ↓
### Bullet cleanup
###        ↓
### Known UI/source noise removal
  ###      ↓
### Whitespace normalization
###        ↓
### Exact duplicate cleanup
###        ↓
### Readable cleaned fields
###        ↓
### Model-oriented symptom text
###        ↓
### Validation
###        ↓
### conditions_text_cleaned.json

##  Next preprocessing component
## The next notebook/module should be symptom parsing.
## That stage will turn symptom descriptions into useful symptom phrases, for example:

### "Symptoms include coughing, wheezing, chest tightness and breathlessness"

## into a structure such as:


### [
###    "coughing",
###    "wheezing",
###    "chest tightness",
###    "breathlessness",
### ]